In [ ]:
# # 1. ouid 목록을 set으로 만들기
# basic_ouids = set(df_basic['ouid'])

# # 2. tier ouid 중 basic에 없는 것만 필터링
# missing_ouids = [ouid for ouid in df_recent['ouid'] if ouid not in basic_ouids]

# # 3. 출력
# for ouid in missing_ouids:
#     print(ouid)

# # ouid (유저 식별 key값)는 같으나 닉네임이 다른경우 발생
# # 조인시 문제가 발생될 수 있고, 해당 이유는 닉네임 변경에 따른 문제라 생각됨
# # 따라서 각 데이터에서 user_name은 드랍처리

# df_tier = df_tier.drop('user_name', axis=1)
# df_basic = df_basic.drop('user_name', axis=1)

# df_tier.to_csv('../../collected_data/tier_final.csv',index=False, encoding='utf-8-sig')
# df_basic.to_csv('../../collected_data/basic_final.csv',index=False, encoding='utf-8-sig')

In [1]:
import pandas as pd
import os

In [18]:
df_ouid = pd.read_csv('../../collected_data/ouid.csv')
df_tier = pd.read_csv('../../collected_data/tier_final.csv')
df_basic = pd.read_csv('../../collected_data/basic_final.csv')
df_recent = pd.read_csv('../../collected_data/recent-info_final.csv')
df_rank = pd.read_csv('../../collected_data/rank_info.csv')

In [17]:
df_ouid[df_ouid['ouid']=='5d84519cf2b0e9c3d69ec23a4a859e12']

,user_name,ouid
4,폭력,5d84519cf2b0e9c3d69ec23a4a859e12


In [12]:
df_tier['party_rank_match_score'].describe()

count     825.000000
mean     2128.019394
std       577.453856
min       955.000000
25%      1600.000000
50%      2027.000000
75%      2487.000000
max      3717.000000
Name: party_rank_match_score, dtype: float64

In [13]:
df_tier['solo_rank_match_score'].describe()

count     825.000000
mean     2082.369697
std       587.170924
min      1001.000000
25%      1600.000000
50%      1957.000000
75%      2400.000000
max      4156.000000
Name: solo_rank_match_score, dtype: float64

In [ ]:
# get_ouid.py를 통해 수집된 data/ouid.csv를 통해
# 캐릭터의 티어 정보 수집
# {
#   "user_name": "string",
#   "solo_rank_match_tier": 0,
#   "solo_rank_match_score": 0,
#   "party_rank_match_tier": 0,
#   "party_rank_match_score": 0
# }

import pandas as pd
import requests
import time
from tqdm import tqdm

with open('./SA/api_request/tier/sa_2.txt', 'r') as f:
    api_key = f.read().strip()

# ouid.csv load
df = pd.read_csv('./SA/collected_data/ouid.csv')
ouids = df['ouid'].dropna().tolist()

results = []

for ouid in tqdm(ouids):
    try:
        url = f"https://open.api.nexon.com/suddenattack/v1/user/tier?ouid={ouid}"
        response = requests.get(
            url,
            headers={"x-nxopen-api-key":api_key}
        )

        if response.status_code == 200:
            data = response.json()
            data['ouid'] = ouid
            results.append(data)
        else:
            print(f'{ouid} - {response.status_code} - {response.text}')

        time.sleep(1.0)

    except Exception as e:
        print(f'[EXCEPTION] {ouid} - {e}')
        time.sleep(1.0)

pd.DataFrame(results).to_csv('./SA/collected_data/tier_info.csv', index=False, encoding='utf-8-sig')


In [24]:
df_rank['ouid'].head(1)

0    9b2852e25a85deebc5ce4173b1ccc1e6
Name: ouid, dtype: object

In [20]:
import pandas as pd
import requests
import time
from tqdm import tqdm

In [22]:
with open('../rank/sa_4.txt', 'r') as f:
    api_key = f.read().strip()

In [31]:
ouid = df_rank['ouid'].head(1).values[0]
url = f"https://open.api.nexon.com/suddenattack/v1/match?ouid={ouid}&match_mode=개인전&match_type=일반전"
response = requests.get(url, headers={"x-nxopen-api-key":api_key})

In [32]:
aa = []
if response.status_code == 200:
    data = response.json()
    data['ouid'] = ouid
    aa.append(data)
else:
    print(f'{ouid} - {response.status_code} - {response.text}')

dd = pd.DataFrame(aa)

In [41]:
for i in dd['match'].tolist():
    print(i[0])

{'match_id': '250726023452007001', 'match_type': '일반전', 'match_mode': '개인전', 'date_match': '2025-07-25T17:34:52.694Z', 'match_result': '2', 'kill': 22, 'death': 29, 'assist': 0}
